# Mistral 7B Architecture from Scratch + Training - PyTorch

## Project Overview

This project explores the implementation of a **Mistral-inspired language model from scratch using PyTorch**, with the goal of understanding the internal architecture of modern decoder-only Large Language Models (LLMs) and training a causal language model on the TinyStories dataset.

Rather than relying on a pretrained Mistral model, the project focuses on building the core Transformer components and implementing the training pipeline to learn how next-token prediction works in practice with an implementation from scratch of the principale components that Mistral introduced (Window Attention , KV-cache(introduced in Llama but used also here) , And for Inference we have the window attention also and the prefill decoding on a fixed size buffer and also for both training ans inference we have RoPE and RMSNorm).

### Architecture and Key Concepts

- **Grouped-Query Attention (GQA):** Shares key-value heads across multiple query heads to reduce memory consumption and improve inference efficiency.
- **Sliding-Window Attention:** Restricts attention to a local context window to reduce attention computation while preserving causal masking.
- **Transformer Architecture:** Explores the attention mechanism, residual connections, normalization, and feed-forward network components of decoder-only language models.
- **Causal Language Modeling:** Trains the model to predict the next token from preceding tokens using an autoregressive objective.
- **PyTorch Training Pipeline:** Implements data preparation, tokenization, forward propagation, loss computation, backpropagation, and parameter optimization.

### Dataset and Training

The model was trained on **TinyStories**, a dataset of short, relatively simple stories designed for language-model training and experimentation.

- **Framework:** PyTorch
- **Training approach:** From-scratch language-model training
- **Training duration:** Approximately 8 hours
- **Objective:** Minimize next-token prediction loss and learn to generate coherent story-like text.
- **Evaluation:** Monitor training loss and evaluate generated text to assess the model's learned language patterns.

### Project Objective

The main objective is to gain hands-on experience with the architecture and training dynamics of modern LLMs, including attention mechanisms, autoregressive generation, optimization, and the practical challenges of training a language model from randomly initialized weights.

*Note: This is a Mistral-inspired educational implementation, not the original pretrained Mistral 7B model. The exact components implemented may differ from the production architecture.*

**Cell overview:** Import PyTorch and the neural-network modules used to implement the model.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

**Cell overview:** Inspect the installed PyTorch/CUDA versions and available GPU hardware.


In [2]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

Torch: 2.11.0+cu128
CUDA: 12.8
GPU: Tesla T4
Capability: (7, 5)


**Cell overview:** Select CUDA acceleration when available; otherwise fall back to CPU.


In [3]:
device = ("cuda" if torch.cuda.is_available() else "cpu")

**Cell overview:** Display the currently selected compute device.


In [4]:
print(device)

cuda


The embedding we are using here a learned embedding not a pretrained one just to stick to principale of implementing everything from scratch

**Cell overview:** Define a trainable token-embedding layer that maps token IDs to hidden vectors.


In [5]:
class Embedding(nn.Module):
    def __init__(self , vocab_size , d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size , d_model)

    def forward(self , x):
        return self.embedding(x)
    

RMSNorm as presented in the Llama Paper we don't need recentering so that's why they removed the mean 

**Cell overview:** Implement RMS normalization using a learnable scaling parameter.


In [6]:
class RMSnorm(nn.Module):
    def __init__(self , d_model , eps = 3e-4):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
    def forward(self ,x):
        # B, T , C = x.shape
        rms = torch.sqrt(torch.mean(x**2 , dim = -1 , keepdim = True) + self.eps)
        x = x/rms
        x = self.weight*x
        return x

In this cell we implement the rotation vectors first we compute the angles for each pair of each head dim then mutliplied them by the position of the token then computed ofr each value its 
cos(theta)+isin(theta)

**Cell overview:** Compute complex rotary-position frequencies for a sequence and attention-head dimension.


In [7]:
def rotation_angles(seq_len , head_dim , theta : float = 10000.0):
    assert head_dim % 2 == 0
    numerators = torch.arange(0 , head_dim , 2)
    thetas = 1 / theta**(numerators/head_dim)
    positions = torch.arange(seq_len)
    freqs = torch.outer(positions , thetas)
    freqs_complex = torch.polar(torch.ones_like(freqs) , freqs)
    return freqs_complex

Here we only reshape the actual pairs of each head dimension to the complex format by regroupin each two consecutive pairs for example [x1 , x2 , x3 , x4] after applying the first line in the next function it becomes [x1 + ix2 , x3 + ix4]

**Cell overview:** Apply rotary positional embeddings (RoPE) to paired query/key features.


In [8]:

def rope(
    x: torch.Tensor,
    seq_len: int,
    head_dim: int,
    device=None,
    start_pos: int = 0
):
    # x: (B, T, n_heads, head_dim)
    x_complex = torch.view_as_complex(
        x.float().reshape(*x.shape[:-1], -1, 2)
    )

    # Generate RoPE angles for the actual absolute positions.
    # During training: start_pos = 0
    # During cached decoding: start_pos may be greater than 0.
    freqs_complex = rotation_angles(
        seq_len,
        head_dim
    ).to(x.device)

    if start_pos != 0:
        # This assumes rotation_angles() generates positions 0..seq_len-1.
        # Recompute the angles for positions start_pos..start_pos+seq_len-1.
        freqs_complex = rotation_angles(
            start_pos + seq_len,
            head_dim
        )[start_pos:start_pos + seq_len].to(x.device)

    # (T, head_dim // 2) -> (1, T, 1, head_dim // 2)
    freqs_complex = freqs_complex.unsqueeze(0).unsqueeze(2)

    # Apply rotary position embeddings.
    x_rotated_complex = x_complex * freqs_complex

    # Convert complex numbers back to real-valued pairs.
    x_out = torch.view_as_real(x_rotated_complex)

    # Restore the input shape and dtype.
    x_out = x_out.reshape(*x.shape)

    return x_out.to(dtype=x.dtype)


**Cell overview:** Import mathematical utilities for attention scaling.


In [9]:
import math

This is very important we are using grouped Query attention so 12 query heads but only 4 kv heads that why here we are duplicating the KV heads to be equal to 12 but on the fly just temporarily on the memory

**Cell overview:** Expand grouped key/value heads to match the number of query heads.


In [10]:
def repeat_kv(x, n_rep):
    # x shape: (B, n_kv_heads, T, head_dim)
    B, n_kv_heads, T, head_dim = x.shape

    if n_rep == 1:
        return x

    x = x[:, :, None, :, :]

    # (B, n_kv_heads, n_rep, T, head_dim)
    x = x.expand(B, n_kv_heads, n_rep, T, head_dim)

    # (B, n_kv_heads * n_rep, T, head_dim)
    return x.reshape(B, n_kv_heads * n_rep, T, head_dim)

This is the Mistral Attention Block it is implemented in hands if you have any question reach out to me

**Cell overview:** Implement causal sliding-window grouped-query attention, including a rolling KV-cache path for decoding.


In [11]:
class MistralAttention(nn.Module):
    def __init__(self , d_model , n_heads , n_kv_heads , window_size):
        super().__init__()
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.window_size = window_size
        self.head_dim = self.d_model // self.n_heads
        self.group_size = self.n_heads // self.n_kv_heads

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        assert window_size > 0
        
        self.Wq = nn.Linear(self.d_model , self.n_heads*self.head_dim , bias = False)
        self.Wk = nn.Linear(self.d_model , self.n_kv_heads*self.head_dim , bias = False)
        self.Wv = nn.Linear(self.d_model , self.n_kv_heads*self.head_dim , bias = False)
        
        self.Wo = nn.Linear(self.d_model , self.d_model , bias = False)

        self.cache_k = None
        self.cache_v = None

    def forward(self , x):
        B ,T , C = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        W = self.window_size

        Q = Q.view(B , T , self.n_heads ,self.head_dim)
        K = K.view(B , T , self.n_kv_heads , self.head_dim)
        V = V.view(B , T , self.n_kv_heads , self.head_dim)

        Q = rope(Q , T , self.head_dim , device = x.device)
        K = rope(K , T , self.head_dim , device = x.device)

        Q = Q.transpose(1,2)
        K = K.transpose(1,2)
        V = V.transpose(1,2)

        K = repeat_kv(K , self.group_size)
        V = repeat_kv(V , self.group_size)

        attn = torch.matmul(Q , K.transpose(-1,-2))
        attn = attn / math.sqrt(self.head_dim)

        positions = torch.arange(T , device = x.device)
        distance = (positions[: , None]-positions[None,:])
        mask = (distance >= 0) & (distance < W)
        attn = attn.masked_fill(~mask , float("-inf"))
        attn = F.softmax(attn , dim = -1)
        scores = torch.matmul(attn , V)
        scores = scores.transpose(1,2).contiguous()
        scores = scores.view(B , T , C)
        out = self.Wo(scores)

        return out


    def forward_chunk(self, x, start_pos):
        B, T, C = x.shape
        W = self.window_size
    
        # Project input into Q, K, V
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
    
        Q = Q.view(B, T, self.n_heads, self.head_dim)
        K = K.view(B, T, self.n_kv_heads, self.head_dim)
        V = V.view(B, T, self.n_kv_heads, self.head_dim)
    
        # RoPE uses absolute positions during cached decoding
        Q = rope(Q, T, self.head_dim, start_pos=start_pos)
        K = rope(K, T, self.head_dim, start_pos=start_pos)
    
        # (B, heads, T, head_dim)
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
    
        # Allocate a rolling cache with W slots
        if self.cache_k is None:
            if start_pos != 0:
                raise ValueError("The KV cache must start at position 0.")
    
            self.cache_k = torch.zeros(
                B, self.n_kv_heads, W, self.head_dim,
                device=x.device, dtype=K.dtype
            )
            self.cache_v = torch.zeros(
                B, self.n_kv_heads, W, self.head_dim,
                device=x.device, dtype=V.dtype
            )
    
        if self.cache_k.shape[0] != B:
            raise ValueError("Batch size does not match the existing KV cache.")
    
        if self.cache_k.device != x.device:
            raise ValueError("KV cache and input must be on the same device.")
    
        # Get previous positions still relevant to the first query
        old_start = max(0, start_pos - W + 1)
        old_positions = torch.arange(
            old_start, start_pos, device=x.device
        )
        old_slots = old_positions % W
    
        old_k = torch.index_select(self.cache_k, 2, old_slots)
        old_v = torch.index_select(self.cache_v, 2, old_slots)
    
        # Combine previous cache entries with the current chunk
        all_k = torch.cat([old_k, K], dim=2)
        all_v = torch.cat([old_v, V], dim=2)
    
        new_positions = torch.arange(
            start_pos, start_pos + T, device=x.device
        )
        key_positions = torch.cat([old_positions, new_positions])
    
        # Expand grouped KV heads to match query heads
        all_k = repeat_kv(all_k, self.group_size)
        all_v = repeat_kv(all_v, self.group_size)
    
        # Scaled dot-product attention
        attn = torch.matmul(Q, all_k.transpose(-1, -2))
        attn = attn / math.sqrt(self.head_dim)
    
        distance = new_positions[:, None] - key_positions[None, :]
        mask = (distance >= 0) & (distance < W)
    
        attn = attn.masked_fill(
            ~mask[None, None, :, :],
            float("-inf")
        )
        attn = F.softmax(attn, dim=-1)
    
        scores = torch.matmul(attn, all_v)
    
        # (B, heads, T, head_dim) -> (B, T, d_model)
        scores = scores.transpose(1, 2).contiguous()
        scores = scores.view(B, T, C)
    
        out = self.Wo(scores)
    
        # Update the rolling cache with the current K and V
        with torch.no_grad():
            for i in range(T):
                slot = (start_pos + i) % W
                self.cache_k[:, :, slot, :] = K[:, :, i, :]
                self.cache_v[:, :, slot, :] = V[:, :, i, :]

        return out

**Cell overview:** Define the position-wise feed-forward network with a SiLU activation.


In [12]:
class FFN(nn.Module):
    def __init__(self , d_model):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(d_model ,4*d_model),
            nn.SiLU(),
            nn.Linear(4*d_model , d_model)
        )

    def forward(self , x):
        return self.ffn(x)
        

**Cell overview:** Combine attention and feed-forward sublayers using pre-normalization and residual connections.


In [13]:
class MistralTransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, window_size):
        super().__init__()

        self.rmsn1 = RMSnorm(d_model)
        self.attention = MistralAttention(d_model, n_heads, n_kv_heads, window_size)

        self.rmsn2 = RMSnorm(d_model)
        self.ffn = FFN(d_model)

    def forward(self, x):
        x = x + self.attention(self.rmsn1(x))
        x = x + self.ffn(self.rmsn2(x))
        return x

**Cell overview:** Assemble the decoder-only language model from embeddings, Transformer blocks, final normalization, and output logits.


In [14]:
class Mistral7B(nn.Module):
    def __init__(self , vocab_size , d_model , n_heads , n_kv_heads , window_size , n_layers = 32):
        super().__init__()
        self.embedding = Embedding(vocab_size ,d_model)
        self.layers = nn.ModuleList([
            MistralTransformerBlock(
                d_model,
                n_heads,
                n_kv_heads,
                window_size
            )
            for _ in range(n_layers)
        ])
        self.rmsx = RMSnorm(d_model)
        self.out = nn.Linear(d_model , vocab_size)

    def forward(self, x):
        # x contains integer token IDs
        x = self.embedding(x)  # Convert token IDs to float vectors
    
        for layer in self.layers:
            x = layer(x)       # Pass hidden states, not token IDs
    
        x = self.rmsx(x)
        out = self.out(x)

        return out

**Cell overview:** Import pandas to load and preprocess the TinyStories CSV files.


In [15]:
import pandas as pd

**Cell overview:** Load the GPT-2 tokenizer used to convert stories into token IDs.


In [16]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

**Cell overview:** Read the TinyStories validation split from its Kaggle dataset path.


In [17]:
validation_dataset = pd.read_csv("/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv")

**Cell overview:** Read the TinyStories training split from its Kaggle dataset path.


In [18]:
df = pd.read_csv("/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/train.csv")

**Cell overview:** Inspect the dimensions of the loaded validation table.


In [19]:
validation_dataset.shape

(21990, 1)

**Cell overview:** Sample 5,000 examples from the validation table for evaluation.


In [20]:
validation_dataset = validation_dataset.sample(5000)

**Cell overview:** Sample 150,000 examples from the training table.


In [21]:
dataset = df.sample(150000)

**Cell overview:** Run Python garbage collection to reclaim unreferenced objects.


In [22]:

import gc
gc.collect()

60

**Cell overview:** Inspect system RAM usage before text processing.


In [23]:
import psutil

ram = psutil.virtual_memory()

print(f"RAM used: {ram.used / 1024**3:.2f} GB")
print(f"RAM available: {ram.available / 1024**3:.2f} GB")
print(f"RAM usage: {ram.percent}%")

RAM used: 3.67 GB
RAM available: 27.23 GB
RAM usage: 13.1%


**Cell overview:** Remove training rows with missing text and ensure the text column contains strings.


In [24]:
dataset = dataset.dropna(subset = ["text"])
dataset["text"] = dataset["text"].astype(str)

**Cell overview:** Clean the validation text column in the same way as the training column.


In [25]:
validation_dataset = validation_dataset.dropna(subset = ["text"])
validation_dataset["text"] = validation_dataset["text"].astype(str)

**Cell overview:** Concatenate the sampled training stories into one text corpus.


In [26]:
all_tokens = "".join(dataset["text"].tolist())

**Cell overview:** Preview the beginning of the concatenated training corpus.


In [27]:
all_tokens[:13]

'One day, a li'

**Cell overview:** Concatenate the sampled validation stories into a separate text corpus.


In [28]:
validation_tokens = "".join(validation_dataset["text"].tolist())

**Cell overview:** Tokenize the training corpus with the GPT-2 tokenizer.


In [29]:
tokens = tokenizer.encode(all_tokens)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (33419846 > 1024). Running this sequence through the model will result in indexing errors


**Cell overview:** Tokenize the validation corpus with the same tokenizer.


In [30]:
validation_tokens = tokenizer.encode(validation_tokens)

**Cell overview:** Check the number of validation token IDs.


In [31]:
len(validation_tokens)

1087659

**Cell overview:** Check the number of training token IDs.


In [32]:
len(tokens)

33419846

**Cell overview:** Import the PyTorch Dataset base class.


In [33]:
from torch.utils.data import Dataset

**Cell overview:** Define fixed-length autoregressive input/next-token target pairs from a token stream.


In [34]:
class TextDataset(Dataset):
    def __init__(self , tokens , context_length):
        self.tokens = torch.tensor(tokens , dtype = torch.long)
        self.context_length = context_length

    def __len__(self):
        return len(self.tokens) // self.context_length

    def __getitem__(self, idx):
        start = idx * self.context_length
        x = self.tokens[start : start + self.context_length]
        y = self.tokens[start + 1 : start + self.context_length + 1]
        return x, y 

**Cell overview:** Construct the training Dataset and a shuffled DataLoader with worker prefetching.


In [35]:
from torch.utils.data import DataLoader

dataset = TextDataset(tokens , 128)

# reduce batch size
train_loader = DataLoader(
    dataset,
    batch_size=24,
    shuffle=True,
    pin_memory=True,
    num_workers=4,      
    prefetch_factor=2,  
    persistent_workers=True  
)

**Cell overview:** Set the model hyperparameters and instantiate the Mistral-inspired Transformer.


In [36]:
vocab_size = tokenizer.vocab_size  # Must match your token IDs

d_model = 768
n_heads = 12
n_kv_heads = 4
window_size = 64

model = Mistral7B(
    vocab_size,
    d_model,
    n_heads,
    n_kv_heads,
    window_size
)

**Cell overview:** Check GPU availability, model vocabulary size, and training-token ID ranges.


In [37]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

print("Model vocabulary size:",
      model.embedding.embedding.num_embeddings)

x, y = next(iter(train_loader))

print("Input range:", x.min().item(), x.max().item())
print("Target range:", y.min().item(), y.max().item())

vocab_size = model.embedding.embedding.num_embeddings

assert x.min().item() >= 0
assert x.max().item() < vocab_size
assert y.min().item() >= 0
assert y.max().item() < vocab_size

print("Token IDs are valid.")

CUDA available: True
Number of GPUs: 2
0 Tesla T4
1 Tesla T4
Model vocabulary size: 50257
Input range: 0 47193
Target range: 0 47193
Token IDs are valid.


**Cell overview:** Create a validation DataLoader. Note: as written, it incorrectly uses `dataset` rather than `validation_dataset`.


In [38]:
validation_dataset = TextDataset(validation_tokens , 128)

# reduce batch size
val_loader = DataLoader(
    dataset,
    batch_size=20,
    shuffle=True,
    pin_memory=True,
    num_workers=4,      
    prefetch_factor=2,  
    persistent_workers=True  
)

**Cell overview:** Inspect GPU memory and utilization through NVIDIA System Management Interface.


In [39]:
!nvidia-smi

Sun Oct 18 13:59:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P0             26W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Cell overview:** Provide a helper to clear attention KV caches between text-generation runs.


In [40]:
def reset_kv_cache(model):
    for layer in model.layers:
        layer.attention.cache_k = None
        layer.attention.cache_v = None

**Cell overview:** Define temperature-controlled autoregressive sampling using cached, sliding-window attention.


In [41]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0):

    was_training = model.training
    model.eval()
    device = next(model.parameters()).device

    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")

    input_ids = tokenizer.encode(
        prompt, return_tensors="pt"
    ).to(device)

    generated = input_ids.clone()
    prompt_len = input_ids.shape[1]

    if prompt_len == 0:
        raise ValueError("Prompt must contain at least one token.")

    if max_new_tokens < 0:
        raise ValueError("max_new_tokens must be >= 0.")

    reset_kv_cache(model)

    try:
        # 1. Prefill the prompt in chunks
        window_size = model.layers[0].attention.window_size
        chunk_size = window_size
        next_logits = None

        for start in range(0, prompt_len, chunk_size):
            end = min(start + chunk_size, prompt_len)
            chunk_ids = input_ids[:, start:end]

            x = model.embedding(chunk_ids)

            for layer in model.layers:
                x = x + layer.attention.forward_chunk(
                    layer.rmsn1(x),
                    start_pos=start
                )
                x = x + layer.ffn(layer.rmsn2(x))

            x = model.rmsx(x)
            logits = model.out(x)
            next_logits = logits[:, -1, :]

        # Absolute position of the next token
        current_pos = prompt_len

        # 2. Generate tokens
        for _ in range(max_new_tokens):
            probs = F.softmax(next_logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            generated = torch.cat([generated, next_token], dim=1)

            # No need to process another token after final prediction
            if generated.shape[1] - prompt_len >= max_new_tokens:
                break

            x = model.embedding(next_token)

            for layer in model.layers:
                x = x + layer.attention.forward_chunk(
                    layer.rmsn1(x),
                    start_pos=current_pos
                )
                x = x + layer.ffn(layer.rmsn2(x))

            x = model.rmsx(x)
            next_logits = model.out(x)[:, -1, :]
            current_pos += 1

        return tokenizer.decode(
            generated[0].tolist(),
            skip_special_tokens=True
        )

    finally:
        reset_kv_cache(model)
        model.train(was_training)

**Cell overview:** Define validation, the mixed-precision training loop, checkpoint selection, and epoch-end text generation.


In [42]:

import time
import math
import os
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm


@torch.no_grad()
def evaluate(model, val_loader, device, epoch=None):
    model.eval()

    total_loss = 0.0
    total_tokens = 0
    start_time = time.time()

    print("\n" + "=" * 65, flush=True)
    print(f"STARTING VALIDATION — Epoch {epoch}", flush=True)
    print(f"Validation batches: {len(val_loader)}", flush=True)
    print("=" * 65, flush=True)

    progress = tqdm(
        val_loader,
        desc=f"Validation {epoch}",
        leave=True,
        dynamic_ncols=True
    )

    for step, (x, y) in enumerate(progress, start=1):
        batch_start = time.time()

        x = x.to(device)
        y = y.to(device)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(x)

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
                reduction="sum"
            )

        total_loss += loss.item()
        total_tokens += y.numel()

        avg_loss = total_loss / total_tokens
        batch_time = time.time() - batch_start
        elapsed = time.time() - start_time

        progress.set_postfix(
            loss=f"{avg_loss:.4f}",
            batch_sec=f"{batch_time:.2f}",
            elapsed_min=f"{elapsed / 60:.1f}"
        )

        del logits, loss

    val_loss = (
        total_loss / total_tokens
        if total_tokens > 0
        else float("nan")
    )

    print("\n" + "-" * 65, flush=True)
    print(f"VALIDATION FINISHED — Epoch {epoch}", flush=True)
    print(f"Validation loss: {val_loss:.4f}", flush=True)
    print(f"Tokens evaluated: {total_tokens:,}", flush=True)
    print(
        f"Validation time: {(time.time() - start_time) / 60:.2f} min",
        flush=True
    )
    print("-" * 65, flush=True)

    return val_loss


def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    tokenizer,
    prompt,
    device,
    epochs=10,
    max_new_tokens=60,
    save_path="best_model.pt",
    log_every=1000
):
    device = torch.device(device)
    model = model.to(device)

    base_model = (
        model.module
        if isinstance(
            model,
            (
                torch.nn.DataParallel,
                torch.nn.parallel.DistributedDataParallel
            )
        )
        else model
    )

    use_amp = device.type == "cuda"

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )

    best_val_loss = float("inf")

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_perplexity": [],
        "learning_rate": [],
        "epoch_time_sec": []
    }

    total_train_batches = len(train_loader)

    print("\n" + "#" * 65, flush=True)
    print("TRAINING STARTED", flush=True)
    print(f"Device: {device}", flush=True)
    print(f"Mixed precision: {use_amp}", flush=True)
    print(f"Epochs: {epochs}", flush=True)
    print(f"Training batches per epoch: {total_train_batches:,}", flush=True)
    print(f"Validation batches: {len(val_loader):,}", flush=True)
    print(f"Checkpoint: {os.path.abspath(save_path)}", flush=True)
    print("#" * 65, flush=True)

    for epoch in range(epochs):
        model.train()

        epoch_start = time.time()
        train_start = time.time()

        total_loss = 0.0
        total_tokens = 0

        print(
            f"\n\n{'=' * 65}\n"
            f"EPOCH {epoch + 1}/{epochs} — TRAINING\n"
            f"{'=' * 65}",
            flush=True
        )

        progress = tqdm(
            train_loader,
            desc=f"Training {epoch + 1}/{epochs}",
            leave=True,
            dynamic_ncols=True
        )

        for step, (x, y) in enumerate(progress, start=1):
            batch_start = time.time()

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp
            ):
                logits = model(x)

                loss = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)),
                    y.reshape(-1)
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )

            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * y.numel()
            total_tokens += y.numel()

            avg_train_loss = total_loss / total_tokens
            batch_time = time.time() - batch_start
            elapsed = time.time() - train_start

            progress.set_postfix(
                loss=f"{avg_train_loss:.4f}",
                batch_sec=f"{batch_time:.2f}",
                elapsed_min=f"{elapsed / 60:.1f}"
            )

            if step % log_every == 0 or step == 1:
                print(
                    f"[TRAIN] Epoch {epoch + 1}/{epochs} | "
                    f"Batch {step}/{total_train_batches} | "
                    f"Loss={avg_train_loss:.4f} | "
                    f"Batch time={batch_time:.2f}s | "
                    f"Grad norm={float(grad_norm):.3f} | "
                    f"Elapsed={elapsed / 60:.1f} min",
                    flush=True
                )

            del logits, loss

        train_loss = (
            total_loss / total_tokens
            if total_tokens > 0
            else float("nan")
        )

        print(
            f"\n[TRAINING COMPLETE] Epoch {epoch + 1} | "
            f"Train loss={train_loss:.4f} | "
            f"Time={(time.time() - train_start) / 60:.2f} min",
            flush=True
        )

        # -------------------------------
        # VALIDATION
        # -------------------------------
        print("[STAGE] Starting validation...", flush=True)

        val_loss = evaluate(
            model,
            val_loader,
            device,
            epoch=epoch + 1
        )

        val_ppl = (
            math.exp(val_loss)
            if math.isfinite(val_loss) and val_loss < 80
            else float("inf")
        )

        lr = optimizer.param_groups[0]["lr"]

        # -------------------------------
        # UPDATE HISTORY
        # -------------------------------
        elapsed = time.time() - epoch_start

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_perplexity"].append(val_ppl)
        history["learning_rate"].append(lr)
        history["epoch_time_sec"].append(elapsed)

        # -------------------------------
        # CHECKPOINT SAVING
        # -------------------------------
        is_best = (
            math.isfinite(val_loss)
            and val_loss < best_val_loss
        )

        if is_best:
            best_val_loss = val_loss

        if is_best:
            print(
                f"\n[SAVE] Saving best checkpoint to {save_path}...",
                flush=True
            )

            save_start = time.time()

            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": base_model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss,
                    "history": history
                },
                save_path
            )

            print(
                f"[SAVE COMPLETE] "
                f"Time={time.time() - save_start:.2f}s",
                flush=True
            )
        else:
            print(
                "[SAVE] Validation loss did not improve; "
                "keeping the previous best checkpoint.",
                flush=True
            )

        # -------------------------------
        # SAMPLE GENERATION
        # -------------------------------
        print("\n[STAGE] Starting sample generation...", flush=True)
        generation_start = time.time()

        try:
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp
            ):
                text = generate(
                    base_model,
                    tokenizer,
                    prompt,
                    max_new_tokens=max_new_tokens,
                    temperature=0.8
                )

            print("\n[GENERATED TEXT]", flush=True)
            print(text, flush=True)

            print(
                f"[GENERATION COMPLETE] "
                f"Time={time.time() - generation_start:.2f}s",
                flush=True
            )

        except Exception as exc:
            print(
                f"[GENERATION ERROR] {type(exc).__name__}: {exc}",
                flush=True
            )

        finally:
            reset_kv_cache(base_model)

        # -------------------------------
        # EPOCH SUMMARY
        # -------------------------------
        print("\n" + "=" * 65, flush=True)
        print(f"EPOCH {epoch + 1}/{epochs} SUMMARY", flush=True)
        print(f"Train loss: {train_loss:.4f}", flush=True)
        print(f"Val loss:   {val_loss:.4f}", flush=True)
        print(f"Val PPL:    {val_ppl:.2f}", flush=True)
        print(f"LR:         {lr:.2e}", flush=True)
        print(f"Epoch time: {elapsed / 60:.1f} min", flush=True)
        print("=" * 65, flush=True)

    print(
        f"\nTRAINING COMPLETE. Best validation loss: {best_val_loss:.4f}",
        flush=True
    )

    return model, history


**Cell overview:** List the GPUs visible to the current PyTorch process.


In [43]:
import torch

print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_name(i))

Number of GPUs: 2
Tesla T4
Tesla T4


**Cell overview:** Move the model to the compute device, optionally wrap it for multiple GPUs, and initialize AdamW.


In [44]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.1
)

**Cell overview:** Train the model for seven epochs while tracking losses, validation perplexity, checkpoints, and samples.


In [ ]:
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    tokenizer=tokenizer,
    prompt="Once upon a time",
    device="cuda",
    epochs=7,
    max_new_tokens=60,
    save_path="best_model.pt",
    log_every=1000
)


#################################################################
TRAINING STARTED
Device: cuda
Mixed precision: True
Epochs: 7
Training batches per epoch: 10,879
Validation batches: 13,055
Checkpoint: /kaggle/working/best_model.pt
#################################################################


EPOCH 1/7 — TRAINING


Training 1/7:   0%|          | 0/10879 [00:00<?, ?it/s]

[TRAIN] Epoch 1/7 | Batch 1/10879 | Loss=11.0444 | Batch time=3.87s | Grad norm=4.143 | Elapsed=0.1 min
[TRAIN] Epoch 1/7 | Batch 1000/10879 | Loss=3.3002 | Batch time=0.68s | Grad norm=0.654 | Elapsed=11.5 min
[TRAIN] Epoch 1/7 | Batch 2000/10879 | Loss=2.9608 | Batch time=0.68s | Grad norm=0.621 | Elapsed=23.0 min
[TRAIN] Epoch 1/7 | Batch 3000/10879 | Loss=2.7796 | Batch time=0.69s | Grad norm=0.623 | Elapsed=34.4 min
[TRAIN] Epoch 1/7 | Batch 4000/10879 | Loss=2.6574 | Batch time=0.68s | Grad norm=0.592 | Elapsed=45.9 min
[TRAIN] Epoch 1/7 | Batch 5000/10879 | Loss=2.5671 | Batch time=0.68s | Grad norm=0.592 | Elapsed=57.3 min
[TRAIN] Epoch 1/7 | Batch 6000/10879 | Loss=2.4952 | Batch time=0.67s | Grad norm=0.567 | Elapsed=68.8 min
[TRAIN] Epoch 1/7 | Batch 7000/10879 | Loss=2.4367 | Batch time=0.68s | Grad norm=0.546 | Elapsed=80.3 min
[TRAIN] Epoch 1/7 | Batch 8000/10879 | Loss=2.3871 | Batch time=0.68s | Grad norm=0.578 | Elapsed=91.8 min
[TRAIN] Epoch 1/7 | Batch 9000/10879 | L

Validation 1:   0%|          | 0/13055 [00:00<?, ?it/s]


-----------------------------------------------------------------
VALIDATION FINISHED — Epoch 1
Validation loss: 1.8684
Tokens evaluated: 33,419,776
Validation time: 54.06 min
-----------------------------------------------------------------

[SAVE] Saving best checkpoint to best_model.pt...
[SAVE COMPLETE] Time=3.84s

[STAGE] Starting sample generation...

[GENERATED TEXT]
Once upon a time, there was a little girl named Lily. She loved to play with her toys all day long. One day, Lily's mommy asked her to help her with the laundry. Lily was patient and never let her mommy use the laundry. 

While Lily was working, her mommy
[GENERATION COMPLETE] Time=6.51s

EPOCH 1/7 SUMMARY
Train loss: 2.2791
Val loss:   1.8684
Val PPL:    6.48
LR:         3.00e-04
Epoch time: 178.8 min


EPOCH 2/7 — TRAINING


...

Validation 2:   0%|          | 0/13055 [00:00<?, ?it/s]


-----------------------------------------------------------------
VALIDATION FINISHED — Epoch 2
Validation loss: 1.6702
Tokens evaluated: 33,419,776
Validation time: 54.21 min
-----------------------------------------------------------------

[SAVE] Saving best checkpoint to best_model.pt...
[SAVE COMPLETE] Time=4.71s

[STAGE] Starting sample generation...

[GENERATED TEXT]
Once upon a time there was a girl named Jane. Jane was three years old and loved to play.

One day, Jane's mom took her to the park. On the way there, she saw lots of kids playing. She wanted to join them, but they looked too messy.

So Jane asked
[GENERATION COMPLETE] Time=4.60s

EPOCH 2/7 SUMMARY
Train loss: 1.8209
Val loss:   1.6702
Val PPL:    5.31
LR:         3.00e-04
Epoch time: 178.8 min


EPOCH 3/7 — TRAINING


Training 3/7:   0%|          | 0/10879 [00:00<?, ?it/s]

[TRAIN] Epoch 3/7 | Batch 1/10879 | Loss=1.6361 | Batch time=0.71s | Grad norm=0.448 | Elapsed=0.0 min
[TRAIN] Epoch 3/7 | Batch 1000/10879 | Loss=1.6703 | Batch time=0.68s | Grad norm=0.484 | Elapsed=11.5 min
[TRAIN] Epoch 3/7 | Batch 2000/10879 | Loss=1.6752 | Batch time=0.68s | Grad norm=0.477 | Elapsed=23.0 min
[TRAIN] Epoch 3/7 | Batch 3000/10879 | Loss=1.6781 | Batch time=0.68s | Grad norm=0.460 | Elapsed=34.4 min
[TRAIN] Epoch 3/7 | Batch 4000/10879 | Loss=1.6806 | Batch time=0.68s | Grad norm=0.477 | Elapsed=45.9 min
[TRAIN] Epoch 3/7 | Batch 5000/10879 | Loss=1.6815 | Batch time=0.68s | Grad norm=0.471 | Elapsed=57.4 min
[TRAIN] Epoch 3/7 | Batch 6000/10879 | Loss=1.6816 | Batch time=0.68s | Grad norm=0.475 | Elapsed=68.8 min
[TRAIN] Epoch 3/7 | Batch 7000/10879 | Loss=1.6819 | Batch time=0.68s | Grad norm=0.463 | Elapsed=80.3 min
[TRAIN] Epoch 3/7 | Batch 8000/10879 | Loss=1.6829 | Batch time=0.74s | Grad norm=0.438 | Elapsed=91.8 min
[TRAIN] Epoch 3/7 | Batch 9000/10879 | Lo

Validation 3:   0%|          | 0/13055 [00:00<?, ?it/s]

The function below is more simpler, easier to implement and to understand

**Cell overview:** Define a simpler generation routine without the temperature parameter; this replaces the earlier `generate` function when executed.


In [ ]:

import torch
import torch.nn.functional as F


@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50):
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device

    input_ids = tokenizer.encode(
        prompt, return_tensors="pt"
    ).to(device)

    generated = input_ids.clone()
    prompt_len = input_ids.shape[1]

    if prompt_len == 0:
        raise ValueError("Prompt must contain at least one token.")

    if max_new_tokens < 0:
        raise ValueError("max_new_tokens must be >= 0.")

    reset_kv_cache(model)

    try:
        # 1. Prefill the prompt in chunks
        window_size = model.layers[0].attention.window_size
        chunk_size = window_size
        next_logits = None

        for start in range(0, prompt_len, chunk_size):
            end = min(start + chunk_size, prompt_len)
            chunk_ids = input_ids[:, start:end]

            x = model.embedding(chunk_ids)

            for layer in model.layers:
                x = x + layer.attention.forward_chunk(
                    layer.rmsn1(x),
                    start_pos=start
                )
                x = x + layer.ffn(layer.rmsn2(x))

            x = model.rmsx(x)
            logits = model.out(x)
            next_logits = logits[:, -1, :]

        # Absolute position of the next token
        current_pos = prompt_len

        # 2. Generate tokens
        for _ in range(max_new_tokens):
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            generated = torch.cat([generated, next_token], dim=1)

            # No need to process another token after the final prediction
            if generated.shape[1] - prompt_len >= max_new_tokens:
                break

            x = model.embedding(next_token)

            for layer in model.layers:
                x = x + layer.attention.forward_chunk(
                    layer.rmsn1(x),
                    start_pos=current_pos
                )
                x = x + layer.ffn(layer.rmsn2(x))

            x = model.rmsx(x)
            next_logits = model.out(x)[:, -1, :]
            current_pos += 1

        return tokenizer.decode(
            generated[0].tolist(),
            skip_special_tokens=True
        )

    finally:
        reset_kv_cache(model)
        model.train(was_training)
